# 06 · Hardware architecture

The earlier notebooks treated SpiNNaker2 as "a fast parallel simulator." This one opens the box:
what the hardware actually *is* — boards, chips, processing elements, their memory, and how spikes
travel between them. Understanding this explains **why** the mapping and memory limits from
[05 · Debugging & reading the output](./05_debugging.ipynb) exist.

**By the end you'll have a mental model of:**

- the physical hierarchy: **board → chip → QPE → processing element (PE)**,
- what one PE is (a small ARM core + local memory + accelerators) and why neurons-per-core is capped,
- the memory hierarchy (PE-local → on-chip → off-chip DRAM),
- how spikes move: on-chip network (NoC), chip-to-chip, and board-to-board.

> **Info**
>
> This notebook is conceptual — no board needed. The one code cell just reads the hardware constants out of the software so the numbers are real, not memorised.

## 1. The hierarchy: board → chip → QPE → PE

SpiNNaker2 is a *many-core* system organised in four levels:

```
Board
└── Chip                     (1 chip on a single-chip board; 48 chips on a 48-node board)
    └── QPE  (Quad PE)       (38 per chip, in a mesh)
        └── PE               (4 per QPE  →  152 PEs per chip)
```

Two bits of terminology used throughout py-spinnaker2:

- **PE** (Processing Element) **= core** — the two terms are interchangeable. It's the unit that runs
  one population's neuron slice (as in the mapping output in [05 · Debugging](./05_debugging.ipynb)).
- **QPE** (Quad PE) **= 4 PEs** — the quad of cores the chip is tiled from.

Every core therefore has a full address — *which chip, which QPE, which PE in the quad* — which is
exactly the `GlobalPE` addresses you saw in chip logs in [05 · Debugging & reading the output](./05_debugging.ipynb).

Let's read the real numbers out of the software:

In [10]:
from spinn_machine.version.version_spin2 import QUAD_MAP
from spinn_machine.version.version_spin2 import CHIPS_PER_BOARD as SPIN2_CHIPS_PER_BOARD
from spinn_machine.version.version_248 import CHIPS_PER_BOARD as SPIN248_CHIPS_PER_BOARD
from spinnman.scamp_constants import APPLICATION_CORES
from spinnaker2.coordinates import GlobalPE

pes_per_chip = len(QUAD_MAP) - 1  # ids 1..152 are PEs (id 0 is special)
chips_per_48_node_board = len(list(SPIN248_CHIPS_PER_BOARD))
print(f"chips per single-node board:       {len(list(SPIN2_CHIPS_PER_BOARD))}")
print(f"PEs per chip:                      {pes_per_chip}   (= {pes_per_chip // 4} QPEs x 4 PEs)")
print(f"application cores / chip:          {len(APPLICATION_CORES)}   (the rest are reserved for system software)")
print(f"chips per 48-node board:           {chips_per_48_node_board}")
print(f"PEs per 48-node board:             {pes_per_chip * chips_per_48_node_board}")
print(f"application cores / 48-node board: {len(APPLICATION_CORES) * chips_per_48_node_board}")

print()
# the full address of one core on a 48-node board:
addr = GlobalPE(chip_x=1, chip_y=0, quad_x=4, quad_y=5, pe_id=2)
print(f"example core address:              {addr}")

chips per single-node board:       1
PEs per chip:                      152   (= 38 QPEs x 4 PEs)
application cores / chip:          148   (the rest are reserved for system software)
chips per 48-node board:           48
PEs per 48-node board:             7296
application cores / 48-node board: 7104

example core address:              Chip_1_0_QPE_4_5_2


So a single chip has **152 PEs** (38 QPEs × 4), of which **148** are available for your applications — the rest run system and communication tasks. A **48-node board** ties **48 chips**
together, and boards can be combined into larger multiboard machines.

## 2. The processing element (PE)

The four PEs in a QPE are identical. Each one packs:

- an **ARM Cortex-M4F** processor with **single-precision floating-point** hardware;
- **128 KB of local SRAM** — fast but small. The app uses roughly **32 KB for the neuron-model code**
  and **96 KB for data** (neuron state, parameters, incoming synapses, recordings). This is *the*
  constraint behind `max_atoms_per_core`: everything a PE's neuron slice needs each timestep must fit
  here (recall the "What drives a PE's memory use" table in [05 · Debugging & reading the output](./05_debugging.ipynb));
- **accelerators** that offload common neuron-model maths from the ARM core: a **machine-learning
  accelerator** (the MAC / matrix-multiply array used by the conv2d models), a fixed-point **exp / log**
  unit, and **random-number generators**;
- plus the usual microcontroller peripherals — **DMA** engines, an **interrupt controller**, and
  **timers**.

A PE doesn't simulate "a neuron" — it simulates a **slice of a population** (up to `max_atoms_per_core`
neurons), stepping them all forward each time step. That's why bigger populations are split across
more PEs.

#### Why PEs are grouped into quads

Within a QPE the four cores can **see each other's local SRAM**, not just their own. Reaching into a
neighbour's SRAM works but is slower than local access, so each PE keeps its neuron slice's hot data
in its own memory.

## 3. Memory hierarchy

Two tiers, fast-and-small vs. slow-and-large:

| Tier | Where | Size (order of) | Used for |
|------|-------|-----------------|----------|
| PE-local SRAM | inside each PE | 128 KB (96 KB data + 32 KB code) | neuron state, params, synapses, recordings — per core |
| global DRAM | shared by all PE's in a chip | 2 GB (two 1 GB banks) | large data & recordings via `record_on_dram=True` |

The golden rule: **the closer to the PE, the faster but scarcer.** Everything a neuron slice needs
for each time step lives in PE-local memory, which is why recording full `v` traces for many neurons, or
too many incoming synapses per core, runs you out of space — and why `hw.run(..., record_on_dram=True)`
exists to move those recordings to the larger but slower DRAM.

## 4. How spikes travel

A spike is a tiny **packet** carrying a *routing key* with the identity of the firing neuron.

Delivering those packets happens at three scales:

- **On-chip — the Network-on-Chip (NoC).** PEs and QPEs are wired by an on-chip network that routes
  spike packets from the firing PE to every PE that needs them.
- **Chip-to-chip.** The 48 chips on a board are joined by **chip-to-chip links**, so a spike whose
  destination lives on another chip crosses over these links.
- **Board-to-board.** For machines larger than one board, **board-to-board links** connect boards into
  a bigger mesh, extending the same packet-routing scheme across the whole system.

## 5. Choosing a machine

The physical hierarchy maps directly onto the board classes from [04 · Hardware allocation](./04_hardware_allocation.ipynb):

| Board class | Chips | Usable cores | Good for |
|-------------|-------|--------------|----------|
| [`SpiNNaker2Chip`](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/hardware-api.html#spinnaker2.hardware.SpiNNaker2Chip) | 1 | 148 | development, small/medium networks |
| [`SpiNNcloud48NodeBoard`](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/hardware-api.html#spinnaker2.hardware.SpiNNcloud48NodeBoard) | 48 | 7104 | large networks |

You choose the class; the mapper then spreads your populations across whatever cores it exposes, using
the `GlobalPE` addressing above to place each neuron slice and route spikes between slices. Larger
machines are just 48-node boards joined by the board-to-board links from the previous section.

## Recap & what's next

- **board → chip → QPE → PE**: 152 PEs/chip (38 QPEs × 4), 148 usable for your application.
- A **PE** is a small ARM core with **128 KB** of local SRAM + MAC / exp-log / RNG accelerators; it
  simulates a *slice* of a population, which is why `max_atoms_per_core` and the memory limits exist.
- Memory has two tiers: **SRAM** local to each PE (fast, small), and **DRAM** shared across the chip (large, slower).
- Spikes travel as tiny packets — routed on-chip by the NoC, then over chip-to-chip and board-to-board links.


**Next up — [07 · Streaming spikes in real time](./07_streaming.ipynb):** feeding inputs and reading
outputs *live* while the hardware runs, instead of only before and after.